# KrishiNexus · `krishi-bangla-lora-v1`: which GGUF export keeps the trained behaviour?

The Q4_K_M GGUF from the training notebook scores below the training run on the frozen test split, when it runs
through Ollama as the app runs it. Flood alerts drop from 1.000 to 0.757 (TRAINING_PLAN.md section 5). This notebook
re-exports the same adapter three ways and scores each GGUF the same way, with the training notebook's own scorer:

- **A-Q8_0:** the adapter merged into the bf16 base, as before, as an 8-bit GGUF.
- **B-Q8_0:** the adapter merged into the base de-quantised from the 4-bit NF4 weights it was trained against, as an
  8-bit GGUF.
- **B-Q4_K_M:** the same merge as a 4-bit GGUF.

The weights are the same in every export. Only the merge and the quantisation change, so scoring them on the test split
changes no training decision.

**Needs:** GPU T4, Internet on, and these inputs: the *KrishiNexus Bangla Agri-Advisor SFT* dataset and the output
of the `krishinexus-bangla-lora-training` notebook.

**Outputs:** the three GGUFs, one `eval_report_<export>.json` and predictions file each, and `export_manifest.json`.


In [ ]:
%pip install -q -U "transformers>=5.22,<6" "peft>=0.17" "accelerate>=1.0" "bitsandbytes>=0.50.2"
# Kaggle ships torchao 0.10; PEFT refuses any torchao older than 0.16 when it merges, and nothing here uses it.
%pip uninstall -y -q torchao


In [ ]:
import gc, hashlib, json, os, shutil, subprocess, sys, time, traceback, urllib.request
from pathlib import Path
import torch

WORK = Path("/kaggle/working")
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_PROMPT = "You are KrishiNexus, a Bangla-speaking agricultural market and storage advisor."   # training/prepare_data.py
OLLAMA = "http://127.0.0.1:11434"
ADAPTER_DIR = next(Path("/kaggle/input").glob("**/krishi-bangla-lora-v1/adapter_config.json")).parent
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
ERRORS = {}                                          # one failed export must not cost the others their results
print("adapter:", ADAPTER_DIR, "| GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Data checks, deterministic scorers (no LLM judge, TRAINING_PLAN.md section 6) and the report. Pure Python, no GPU.
import json
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path

SPLITS = ("train", "val", "test")
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
VERDICTS = ("BLOCK_SPOT_SALE", "ALLOW_STORAGE", "ALLOW_SPOT_SALE", "WARN_STORAGE_LOSES_VALUE", "WARN_NO_STORAGE_CAPACITY")
ALERTS = {"RED": "RED", "ORANGE": "ORANGE", "YELLOW": "YELLOW", "GREEN": "GREEN",
          "লাল": "RED", "কমলা": "ORANGE", "হলুদ": "YELLOW", "সবুজ": "GREEN"}
PRICE_RE = re.compile(r"খামার দাম কেজি (\d+) টাকা, পাইকারি (\d+) টাকা, খুচরা (\d+) টাকা")
TRACE_RE = re.compile(r"storage = [^;]*?= (-?[\d,]+); spot = [^;]*?= (-?[\d,]+); dPi = (-?[\d,]+)")
PCT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*%")
CITE_RE = re.compile(r"\[([A-Z][A-Z0-9]*(?:-[A-Z0-9]+)+)\]")
UNFAIR = ("দামের চেয়ে", "ন্যায্য নয়", "ন্যায্য না", "অন্যায্য", "বিক্রি না করে", "বিক্রি করবেন না", "not fair", "unfair", "too low")
FAIR = ("ন্যায্য", "বিক্রি করতে পারেন", "fair")
TARGETS = {  # TRAINING_PLAN.md section 6
    "negotiation_decision_accuracy": 0.90, "negotiation_price_fidelity": 0.85,
    "arbitrage_verdict_exact_match": 0.90, "arbitrage_trace_fidelity": 0.80,
    "flood_alert_accuracy": 0.90, "flood_probability_within_2pts": 0.85,
    "storage_citation_accuracy": 0.90, "storage_chrf": 60.0,
}
LOWER_IS_BETTER = {"negotiation_unparsed_rate", "flood_probability_mae_pts"}


def find_data_dir():
    """An attached Kaggle dataset (any mount layout) or a local training/data folder; None if neither exists."""
    candidates = sorted(Path("/kaggle/input").glob("**/split_report.json")) if Path("/kaggle/input").exists() else []
    candidates += [Path(p) / "split_report.json" for p in ("training/data", "../training/data", "data")]
    for report in candidates:
        if report.exists() and all((report.parent / f"{s}.jsonl").exists() for s in SPLITS):
            return report.parent
    return None


def load_splits(data_dir):
    """Load the three JSONL splits and fail loudly on anything that would make training or scoring wrong."""
    data_dir = Path(data_dir)
    rows = {s: [json.loads(line) for line in (data_dir / f"{s}.jsonl").open(encoding="utf-8") if line.strip()] for s in SPLITS}
    report = json.loads((data_dir / "split_report.json").read_text(encoding="utf-8"))
    for split, n in report["splits"].items():
        assert len(rows[split]) == n, f"{split}.jsonl has {len(rows[split])} rows but split_report.json says {n}"
    ids = {s: {r["example_id"] for r in rows[s]} for s in SPLITS}
    assert not (ids["train"] & ids["val"] or ids["train"] & ids["test"] or ids["val"] & ids["test"]), "an example_id is in two splits"
    # Answer families (identical category + answer) never cross splits; negotiation is split by market cell upstream.
    families = {s: {(r["category"], answer(r)) for r in rows[s] if r["category"] != "market_negotiation"} for s in SPLITS}
    assert not (families["train"] & families["val"] or families["train"] & families["test"] or families["val"] & families["test"]), \
        "an answer family straddles splits"
    for split in SPLITS:
        for r in rows[split]:
            assert [m["role"] for m in r["messages"]] == ["system", "user", "assistant"], f"{r['example_id']}: unexpected roles"
    return rows, report


def check_reference_formats(rows):
    """Every reference must parse, or the scorers below would silently mis-score."""
    for r in rows:
        ref, task = answer(r), r["category"]
        ok = {"market_negotiation": lambda: decision(ref) and PRICE_RE.search(ref),
              "arbitrage_math": lambda: verdict(ref) and trace(ref),
              "flood_harvest": lambda: alert(ref) and first_pct(ref) is not None,
              "storage_qa": lambda: len(citations(ref)) == 1}[task]()
        assert ok, f"{r['example_id']}: reference is not in the expected {task} format"


def answer(row):
    return row["messages"][-1]["content"]


def question(row):
    """The farmer's words without the tool result (prompts v4 append one), which a prompt-blind baseline may not read."""
    return row["messages"][1]["content"].split("\n<tool_result")[0]


def to_prompt_completion(row):
    """TRL conversational prompt-completion row: loss is computed on the assistant turn only."""
    return {"prompt": row["messages"][:-1], "completion": row["messages"][-1:]}


def token_lengths(tokenizer, rows):
    return [len(tokenizer(tokenizer.apply_chat_template(r["messages"], tokenize=False), add_special_tokens=False)["input_ids"])
            for r in rows]


def eval_subset(rows, limit):
    """Deterministic stride sample that keeps every task (the file is ordered by task); all rows when limit is None."""
    if not limit or limit >= len(rows):
        return rows
    return rows[:: len(rows) // limit][:limit]


def norm(text):
    return text.translate(BN_DIGITS)


def ints(text):
    return {int(x) for x in re.findall(r"\d+", norm(text))}


def earliest(text, vocab):
    hits = [(text.find(v), v) for v in vocab if v in text]
    return min(hits)[1] if hits else None


def decision(text):
    t = norm(text).lower()
    return "unfair" if any(c in t for c in UNFAIR) else "fair" if any(c in t for c in FAIR) else None


def verdict(text):
    return earliest(text, VERDICTS)


def alert(text):
    hit = earliest(text, tuple(ALERTS))
    return ALERTS[hit] if hit else None


def first_pct(text):
    match = PCT_RE.search(norm(text))
    return float(match.group(1)) if match else None


def trace(text):
    match = TRACE_RE.search(norm(text))
    return tuple(int(v.replace(",", "")) for v in match.groups()) if match else None


def citations(text):
    return set(CITE_RE.findall(text))


def uncited(text):
    return CITE_RE.sub("", text).strip()


def close(got, gold):
    return abs(got - gold) <= max(1.0, 0.005 * abs(gold))


def chrf(hyps, refs, order=6, beta=2.0):
    """Corpus chrF2 as in sacreBLEU's default: char n-grams 1-6, whitespace removed, P and R averaged over orders."""
    stats = [[0, 0, 0] for _ in range(order)]  # hypothesis n-grams, reference n-grams, matches
    for hyp, ref in zip(hyps, refs):
        hyp, ref = "".join(hyp.split()), "".join(ref.split())
        for n in range(1, order + 1):
            h = Counter(hyp[i:i + n] for i in range(len(hyp) - n + 1))
            r = Counter(ref[i:i + n] for i in range(len(ref) - n + 1))
            stats[n - 1][0] += sum(h.values())
            stats[n - 1][1] += sum(r.values())
            stats[n - 1][2] += sum((h & r).values())
    eps = 1e-16
    precision = sum(m / h if h else eps for h, _, m in stats) / order
    recall = sum(m / r if r else eps for _, r, m in stats) / order
    factor = beta ** 2
    return 100 * (1 + factor) * precision * recall / (factor * precision + recall) if precision + recall else 0.0


def score_example(row, pred):
    """Per-example 0/1 scores for the row's task, keyed by report metric name."""
    ref, task = answer(row), row["category"]
    if task == "market_negotiation":
        return {"negotiation_decision_accuracy": decision(pred) == decision(ref),
                "negotiation_price_fidelity": set(map(int, PRICE_RE.search(ref).groups())) <= ints(pred)}
    if task == "arbitrage_math":
        got, gold = trace(pred), trace(ref)
        return {"arbitrage_verdict_exact_match": verdict(pred) == verdict(ref),
                "arbitrage_trace_fidelity": got is not None and all(close(a, b) for a, b in zip(got, gold))}
    if task == "flood_harvest":
        got = first_pct(pred)
        return {"flood_alert_accuracy": alert(pred) == alert(ref),
                "flood_probability_within_2pts": got is not None and abs(got - first_pct(ref)) <= 2}
    return {"storage_citation_accuracy": citations(pred) == citations(ref)}


def aggregate(rows, preds):
    """metric -> {value, n} for 'all' and each test subset; chrF, balanced accuracy and MAE are computed per group."""
    out = {}
    for group in ["all"] + sorted({r["subset"] for r in rows}):
        sel = [(r, p) for r, p in zip(rows, preds) if group == "all" or r["subset"] == group]
        sums, counts = defaultdict(float), Counter()
        for r, p in sel:
            for metric, value in score_example(r, p).items():
                sums[metric] += float(value)
                counts[metric] += 1
        res = {m: {"value": round(sums[m] / counts[m], 4), "n": counts[m]} for m in sorted(counts)}
        pairs = [(decision(answer(r)), decision(p)) for r, p in sel if r["category"] == "market_negotiation"]
        if pairs:
            recalls = []
            for label in ("fair", "unfair"):
                gold_n = sum(g == label for g, _ in pairs)
                if gold_n:
                    recalls.append(sum(g == label and q == label for g, q in pairs) / gold_n)
            res["negotiation_balanced_accuracy"] = {"value": round(sum(recalls) / len(recalls), 4), "n": len(pairs)}
            res["negotiation_unparsed_rate"] = {"value": round(sum(q is None for _, q in pairs) / len(pairs), 4), "n": len(pairs)}
        errors = [abs(first_pct(p) - first_pct(answer(r))) for r, p in sel
                  if r["category"] == "flood_harvest" and first_pct(p) is not None]
        if any(r["category"] == "flood_harvest" for r, _ in sel):
            res["flood_probability_mae_pts"] = {"value": round(statistics.mean(errors), 2) if errors else None, "n": len(errors)}
        storage = [(uncited(p), uncited(answer(r))) for r, p in sel if r["category"] == "storage_qa"]
        if storage:
            res["storage_chrf"] = {"value": round(chrf(*zip(*storage)), 2), "n": len(storage)}
        out[group] = res
    return out


def baseline_predictions(test_rows, train_rows):
    """Prompt-blind baselines built from TRAIN only: the majority label per task, and a train-answer lookup for storage."""
    def majority(parse, task):
        return Counter(parse(answer(r)) for r in train_rows if r["category"] == task).most_common(1)[0][0]

    negotiation = {"unfair": "ন্যায্য খামার দামের চেয়ে কম।", "fair": "এই দাম ন্যায্য, বিক্রি করতে পারেন।"}[majority(decision, "market_negotiation")]
    arbitrage, flood = majority(verdict, "arbitrage_math"), f"({majority(alert, 'flood_harvest')})"
    by_question = defaultdict(list)
    for r in train_rows:
        if r["category"] == "storage_qa":
            by_question[question(r)].append(answer(r))

    def storage(question):
        options = by_question.get(question)
        if not options:
            return ""
        doc = Counter(next(iter(citations(a))) for a in options).most_common(1)[0][0]
        return next(a for a in options if doc in citations(a))

    fixed = {"market_negotiation": negotiation, "arbitrage_math": arbitrage, "flood_harvest": flood}
    return [fixed.get(r["category"]) or storage(question(r)) for r in test_rows]


def build_report(test_rows, preds, train_rows, meta):
    """preds = {"adapter": [...], "base": [...]} in test_rows order; adds the prompt-blind baseline, targets and deltas."""
    results = {name: aggregate(test_rows, p) for name, p in preds.items()}
    results["baseline"] = aggregate(test_rows, baseline_predictions(test_rows, train_rows))

    def overall(name, metric):
        return results.get(name, {}).get("all", {}).get(metric, {}).get("value")

    summary = []
    for metric in sorted(results["baseline"]["all"]):
        a, b, z = overall("adapter", metric), overall("base", metric), overall("baseline", metric)
        better = None if a is None or b is None else (a < b if metric in LOWER_IS_BETTER else a > b)
        summary.append({"metric": metric, "target": TARGETS.get(metric), "adapter": a, "base": b, "baseline": z,
                        "improved_over_base": better,
                        "meets_target": None if metric not in TARGETS or a is None else a >= TARGETS[metric]})
    return {**meta, "targets": TARGETS, "summary": summary, "results": results}


def _fmt(value):
    return "–" if value is None else f"{value:.3f}" if isinstance(value, float) else str(value)


def summary_table(report):
    lines = ["| metric | target | adapter | base | baseline | adapter beats base | meets target |", "|---|---|---|---|---|---|---|"]
    for s in report["summary"]:
        name = s["metric"] + (" (lower is better)" if s["metric"] in LOWER_IS_BETTER else "")
        lines.append(f"| {name} | {_fmt(s['target'])} | {_fmt(s['adapter'])} | {_fmt(s['base'])} | {_fmt(s['baseline'])} "
                     f"| {_fmt(s['improved_over_base'])} | {_fmt(s['meets_target'])} |")
    return "\n".join(lines)


def subset_table(report, model="adapter"):
    groups = [g for g in report["results"][model] if g != "all"]
    lines = [f"| {model}: metric | " + " | ".join(groups) + " |", "|---|" + "---|" * len(groups)]
    for metric in sorted(report["results"][model]["all"]):
        cells = [report["results"][model][g].get(metric, {}) for g in groups]
        lines.append(f"| {metric} | " + " | ".join(f"{_fmt(c.get('value'))} (n={c['n']})" if c else "–" for c in cells) + " |")
    return "\n".join(lines)


In [ ]:
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MERGED = {"A": WORK / "merged-a", "B": WORK / "merged-b"}


def save_merged(base, out):
    PeftModel.from_pretrained(base, str(ADAPTER_DIR)).merge_and_unload().save_pretrained(str(out))
    # Training never changes the tokenizer; llama.cpp's converter needs Qwen's original files (as in the training notebook).
    snapshot_download(BASE_MODEL, local_dir=str(out), allow_patterns=["tokenizer*", "vocab.json", "merges.txt"])


try:   # A: what the training notebook exported, the adapter on the bf16 base
    save_merged(AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16), MERGED["A"])
except Exception:
    ERRORS["merge A"] = traceback.format_exc()
try:   # B: the base the adapter was trained against, NF4 with double quantisation and fp16 compute, de-quantised to fp16
    nf4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=nf4, dtype=torch.float16, device_map={"": 0})
    base = base.dequantize() or base               # some transformers versions de-quantise in place
    if hasattr(base.config, "quantization_config"):
        del base.config.quantization_config
    save_merged(base, MERGED["B"])
except Exception:
    ERRORS["merge B"] = traceback.format_exc()
gc.collect()
torch.cuda.empty_cache()
print({k: v[-600:] for k, v in ERRORS.items()} or "both merges saved")


In [ ]:
def run(cmd, **kwargs):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kwargs)


GGUF = {}
try:
    llama = WORK / "llama.cpp"
    run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", llama])
    deps = WORK / "gguf-deps"                        # the converter's pinned packages live here, not in the kernel
    run([sys.executable, "-m", "pip", "install", "-q", "--target", deps, "-r",
         llama / "requirements" / "requirements-convert_hf_to_gguf.txt"])
    env = {**os.environ, "PYTHONPATH": str(deps)}
    LLAMA_COMMIT = subprocess.run(["git", "-C", str(llama), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    for name, (variant, outtype) in {"A-Q8_0": ("A", "q8_0"), "B-Q8_0": ("B", "q8_0"), "B-F16": ("B", "f16")}.items():
        if MERGED[variant].exists():
            out = WORK / f"krishi-bangla-{name.lower()}.gguf"
            run([sys.executable, llama / "convert_hf_to_gguf.py", MERGED[variant], "--outtype", outtype, "--outfile", out], env=env)
            GGUF[name] = out
    if "B-F16" in GGUF:
        if shutil.which("cmake") is None:
            run([sys.executable, "-m", "pip", "install", "-q", "cmake"])
        cmake = shutil.which("cmake") or str(Path(sys.executable).parent / "cmake")
        run([cmake, "-S", llama, "-B", llama / "build", "-DCMAKE_BUILD_TYPE=Release"])
        run([cmake, "--build", llama / "build", "--config", "Release", "--target", "llama-quantize", "-j", os.cpu_count() or 2])
        GGUF["B-Q4_K_M"] = WORK / "krishi-bangla-b-q4_k_m.gguf"
        run([llama / "build" / "bin" / "llama-quantize", GGUF.pop("B-F16"), GGUF["B-Q4_K_M"], "Q4_K_M"])
except Exception:
    ERRORS["convert"] = traceback.format_exc()
for path in [*MERGED.values(), WORK / "gguf-deps", WORK / "krishi-bangla-b-f16.gguf"]:
    shutil.rmtree(path, ignore_errors=True) if path.is_dir() else path.unlink(missing_ok=True)
print({name: f"{path.stat().st_size / 2**30:.2f} GiB" for name, path in GGUF.items()}, ERRORS.get("convert", "")[-600:])


In [ ]:
def call(path, body=None, timeout=600):
    req = urllib.request.Request(OLLAMA + path, data=None if body is None else json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


try:   # Ollama is the runtime the app uses, so each export is scored exactly as the app would call it
    # Ollama now ships as .tar.zst and Kaggle's image has no zstd to unpack it.
    run(["bash", "-c", "(apt-get update -qq || true) && apt-get install -y -qq zstd && curl -fsSL https://ollama.com/install.sh | sh"])
    SERVER = subprocess.Popen(["ollama", "serve"], stdout=open(WORK / "ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(90):
        try:
            OLLAMA_VERSION = call("/api/version")["version"]
            break
        except OSError:
            time.sleep(1)
    print("ollama", OLLAMA_VERSION)
except Exception:
    ERRORS["ollama"] = traceback.format_exc()


In [ ]:
rows, split_report = load_splits(find_data_dir())
TEST, TRAIN = rows["test"], rows["train"]
SCORES, TIMING = {}, {}
for name, path in GGUF.items():
    tag = "krishi-" + name.lower().replace("_", "-")
    try:
        modelfile = WORK / f"Modelfile.{name}"
        modelfile.write_text(f"FROM {path}\nPARAMETER temperature 0.2\nSYSTEM {SYSTEM_PROMPT}\n", encoding="utf-8")
        run(["ollama", "create", tag, "-f", modelfile])
        preds, rates, started = [], [], time.time()
        for row in TEST:
            out = call("/api/generate", {"model": tag, "prompt": row["messages"][1]["content"], "stream": False,
                                         "options": {"temperature": 0, "num_predict": 248, "seed": 0}})
            preds.append(out["response"])
            if out.get("eval_duration"):
                rates.append(out["eval_count"] / (out["eval_duration"] / 1e9))
        TIMING[name] = {"minutes": round((time.time() - started) / 60, 1),
                        "decode_tokens_per_s_median": round(sorted(rates)[len(rates) // 2], 1) if rates else None}
        report = build_report(TEST, {"adapter": preds}, TRAIN, {
            "export": name, "runtime": f"Ollama {OLLAMA_VERSION}", "gguf_bytes": path.stat().st_size, "split": "test",
            "n_examples": len(TEST), "data_sha256": split_report.get("jsonl_sha256"), "timing": TIMING[name],
            "decoding": {"temperature": 0, "num_predict": 248, "seed": 0, "system": "Modelfile SYSTEM line"},
            "note": "'adapter' in the summary is this GGUF export"})
        (WORK / f"eval_report_{name}.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        (WORK / f"predictions_test_{name}.jsonl").write_text("".join(
            json.dumps({"example_id": r["example_id"], "category": r["category"], "subset": r["subset"], "gguf": p},
                       ensure_ascii=False) + "\n" for r, p in zip(TEST, preds)), encoding="utf-8")
        SCORES[name] = {s["metric"]: (s["adapter"], s["meets_target"]) for s in report["summary"]}
        run(["ollama", "rm", tag])
    except Exception:
        ERRORS[f"score {name}"] = traceback.format_exc()
        print(ERRORS[f"score {name}"][-800:])
for metric, target in TARGETS.items():
    print(f"{metric:<34} target {target:<5}", "  ".join(f"{n} {SCORES[n][metric][0]}" for n in SCORES if metric in SCORES[n]))


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 23), b""):
            h.update(block)
    return h.hexdigest()


manifest = {"base_model": BASE_MODEL, "adapter_dir": str(ADAPTER_DIR), "ollama": globals().get("OLLAMA_VERSION"),
            "llama_cpp_commit": globals().get("LLAMA_COMMIT"), "gpu": torch.cuda.get_device_name(0),
            "gguf": {name: {"file": path.name, "bytes": path.stat().st_size, "sha256": sha256(path)} for name, path in GGUF.items()},
            "timing": TIMING, "errors": {k: v[-2000:] for k, v in ERRORS.items()},
            "meets_every_target": {n: all(ok for ok in (m for _, m in SCORES[n].values()) if ok is not None) for n in SCORES}}
(WORK / "export_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
shutil.rmtree(WORK / "llama.cpp", ignore_errors=True)
print(json.dumps({k: manifest[k] for k in ("gguf", "meets_every_target", "errors")}, indent=1)[:4000])
